# 602 Stride prefix sweep: h16 anchor, h8 anchor, then all 34 seed-7 points

Restart the runtime and run all cells. Upload the single Sacramento input archive when prompted. Tiny untrainable prefixes remain explicit statuses.

In [ ]:
from pathlib import Path
import gzip, json, shutil, subprocess, sys, tarfile
from google.colab import files

BRANCH = "experiment/602-stride-live-inference"
REPO_URL = "https://github.com/Angelawoo572/cache_arch.git"
REPO = Path("/content/cache_arch")
RUN_DIR = Path("/content/stride_run")

if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "fetch", "origin"], check=True)
    subprocess.run(["git", "-C", str(REPO), "switch", BRANCH], check=True)
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, str(REPO)], check=True)
print(subprocess.check_output(["git", "-C", str(REPO), "branch", "--show-current"], universal_newlines=True).strip())


In [ ]:
uploaded = files.upload()
archive_names = [name for name in uploaded if name.endswith(".tar.gz")]
assert len(archive_names) == 1, "upload exactly one input archive"
archive = Path("/content") / archive_names[0]
if RUN_DIR.exists():
    raise RuntimeError("/content/stride_run already exists; restart for a clean run")
RUN_DIR.mkdir()
with tarfile.open(archive, "r:gz") as handle:
    for member in handle.getmembers():
        parts = Path(member.name).parts
        assert not Path(member.name).is_absolute() and ".." not in parts
        assert not member.issym() and not member.islnk()
    handle.extractall(RUN_DIR)
for manifest_path in RUN_DIR.glob("training_prefixes/*/training_manifest.json"):
    item = json.loads(manifest_path.read_text())
    stream = manifest_path.parent / Path(item["training_stream"]).name
    assert stream.is_file(), stream
    with gzip.open(stream, "rt") as handle:
        next(handle)
assert len(list(RUN_DIR.glob("training_prefixes/*/training_manifest.json"))) == 17
evaluation = RUN_DIR / "evaluation/602.gcc_s-734B.eval_stream.csv.gz"
with gzip.open(evaluation, "rt") as handle:
    next(handle)
print("input archive structure, 17 prefix streams, and evaluation stream PASS")


In [ ]:
EXP = REPO / "formal_NN_training/experiments/602_lstm_stride_live_inference"
SWEEP = EXP / "python/run_training_prefix_sweep.py"
EVAL = RUN_DIR / "evaluation/602.gcc_s-734B.eval_stream.csv.gz"

def run_sweep(hidden_sizes, budgets):
    command = [sys.executable, str(SWEEP), "--run-dir", str(RUN_DIR), "--evaluation-stream", str(EVAL), "--hidden-sizes", hidden_sizes, "--budgets", budgets, "--seeds", "7", "--device", "auto", "--resume"]
    print(" ".join(command))
    subprocess.run(command, check=True)


## Stage 0: h16 × 20M correctness/performance anchor

In [ ]:
run_sweep("16", "i20m")


## Stage 1: h8 × 20M compact anchor

In [ ]:
run_sweep("8", "i20m")


## Stage 2: complete h8/h16 seed-7 sweep

The runner resumes the anchors and records no-callback, single-class, insufficient, failed, and all-silent points instead of borrowing future rows.

In [ ]:
run_sweep("8,16", "all")


## Export every trained checkpoint

Each export includes model.bin and model_metadata.json; no weight is committed.

In [ ]:
EXPORTER = EXP / "python/export_live_model.py"
for metadata_path in sorted(RUN_DIR.glob("points/h*/*/seed*/point_metadata.json")):
    row = json.loads(metadata_path.read_text())
    checkpoint = metadata_path.parent / "offline/model.pt"
    if not checkpoint.is_file():
        continue
    output = metadata_path.parent / "export"
    if (output / "model.bin").is_file() and (output / "model_metadata.json").is_file():
        continue
    command = [sys.executable, str(EXPORTER), "--checkpoint", str(checkpoint), "--run-metadata", str(metadata_path.parent / "offline/run_metadata.json"), "--out-dir", str(output), "--hidden-size", str(row["hidden_size"]), "--instruction-budget", str(row["instruction_budget"]), "--seed", str(row["seed"])]
    subprocess.run(command, check=True)
print("exports complete")


## Package and download the single Colab output archive

In [ ]:
output_archive = Path("/content/602_stride_live_colab_output.tar.gz")
with tarfile.open(output_archive, "w:gz") as handle:
    handle.add(RUN_DIR / "points", arcname="points")
    status = RUN_DIR / "training_sweep_status.json"
    if status.is_file():
        handle.add(status, arcname="training_sweep_status.json")
print(output_archive, output_archive.stat().st_size)
files.download(str(output_archive))
